# 01 — Exploratory Data Analysis
Visualise raw UCI HAR inertial signals across all 6 activity classes.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.ingestion.data_loader import UCIHARDataLoader
from src.utils.config import Config

cfg = Config()
loader = UCIHARDataLoader(cfg)
X_train, X_test, y_train, y_test = loader.load()

## Dataset overview

In [ ]:
print(f'X_train shape : {X_train.shape}  (samples, timesteps, channels)')
print(f'X_test  shape : {X_test.shape}')
print(f'Classes       : {cfg.activity_labels}')
print(f'Train NaNs    : {np.isnan(X_train).sum()}')

# Class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y, title in zip(axes, [y_train, y_test], ['Train', 'Test']):
    counts = [np.sum(y == c) for c in range(6)]
    ax.bar(cfg.activity_labels.values(), counts, color='steelblue', edgecolor='white')
    ax.set_title(f'{title} class distribution')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## Raw signal plots — one sample per activity class

In [ ]:
CHANNEL_NAMES = [
    'body_acc_x', 'body_acc_y', 'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z',
]
t = np.arange(128) / 50.0

fig, axes = plt.subplots(6, 3, figsize=(15, 18), sharex=True)
fig.suptitle('Raw Accelerometer Channels (body_acc) — One sample per class', fontsize=13, fontweight='bold')

for cls_idx, cls_name in cfg.activity_labels.items():
    idx = np.where(y_train == cls_idx)[0][0]
    for ch in range(3):  # body_acc x/y/z
        ax = axes[cls_idx][ch]
        ax.plot(t, X_train[idx, :, ch], linewidth=0.9)
        ax.set_title(f'{cls_name}\n{CHANNEL_NAMES[ch]}', fontsize=8)
        ax.grid(True, alpha=0.3)
        if ch == 0:
            ax.set_ylabel('Amplitude', fontsize=8)
    axes[cls_idx][-1].set_xlabel('Time (s)', fontsize=8)

plt.tight_layout()
plt.show()

## Signal statistics per class

In [ ]:
import pandas as pd

rows = []
for cls_idx, cls_name in cfg.activity_labels.items():
    mask = y_train == cls_idx
    X_cls = X_train[mask]  # (N_cls, 128, 9)
    rows.append({
        'Activity': cls_name,
        'N': mask.sum(),
        'Mean (body_acc_x)': X_cls[:, :, 0].mean().round(4),
        'Std  (body_acc_x)': X_cls[:, :, 0].std().round(4),
        'Mean (body_gyro_x)': X_cls[:, :, 3].mean().round(4),
        'Std  (body_gyro_x)': X_cls[:, :, 3].std().round(4),
    })

pd.DataFrame(rows).set_index('Activity')

## FFT power spectrum comparison

In [ ]:
from scipy.fft import rfft, rfftfreq

freqs = rfftfreq(128, d=1/50)

fig, ax = plt.subplots(figsize=(12, 5))
for cls_idx, cls_name in cfg.activity_labels.items():
    mask = y_train == cls_idx
    spectrum = np.abs(rfft(X_train[mask, :, 0], axis=1)).mean(axis=0) ** 2
    ax.plot(freqs, spectrum, label=cls_name, linewidth=1.2)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Mean Power')
ax.set_title('Mean FFT Power Spectrum — body_acc_x per activity class')
ax.legend()
ax.set_xlim(0, 25)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()